# Empirical analysis

This notebook provides a guided route through the empirical part of the replication package. It prepares the European Values Study data, summarizes the two environmental-attitude measures, estimates the models reported in the paper, calculates average changes in predicted probabilities, and creates the country-level figures.

The survey data are not included because their licence does not allow redistribution. Follow `data/README.md` and place `ZA7500_v5-0-0.sav` in `data/raw/` before running the full notebook. This notebook runs the analysis scripts in order and presents their main outputs.

## 1. Set up paths and display options

The following cell finds the repository root, whether the notebook is opened from the repository or from the `notebooks/original` folder.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display


def find_repository_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "run_all.py").exists() and (candidate / "code").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the replication repository.")


ROOT = find_repository_root()
DATA_FILE = ROOT / "data" / "raw" / "ZA7500_v5-0-0.sav"
TABLES = ROOT / "output" / "tables"
FIGURES = ROOT / "output" / "figures"

pd.set_option("display.max_columns", 30)
print(f"Repository: {ROOT}")
print(f"Survey data available: {DATA_FILE.exists()}")

## 2. Prepare the analysis sample

The preparation step keeps only the variables used in the paper, removes observations with missing values on the required fields, recodes the two five-point responses into three ordered categories, and calculates the country-level measures used in the figures.

If the data file is missing, consult `data/README.md`.

In [ ]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Survey file not found at {DATA_FILE}. "
        "See data/README.md for access and placement instructions."
    )

subprocess.run([sys.executable, str(ROOT / "code" / "01_prepare_data.py")], check=True)

## 3. Inspect the prepared data

The preview below uses readable labels for the variables most often discussed in the paper. The original EVS variable names remain available in the saved analysis file.

In [ ]:
sample = pd.read_csv(ROOT / "data" / "interim" / "analysis_sample.csv")

preview_columns = [
    "c_abrv", "year", "v31", "v199", "v202", "v261", "v276_r",
    "v199_decoded", "v202_decoded",
]
sample[preview_columns].head()

In [ ]:
summary = pd.DataFrame({
    "observations": [len(sample)],
    "countries": [sample["c_abrv"].nunique()],
    "complete involvement outcome": [sample["v199_decoded"].notna().sum()],
    "complete uncertainty outcome": [sample["v202_decoded"].notna().sum()],
})
summary

## 4. Descriptive relationship between the two attitude measures

The first measure records willingness to give part of one's income to prevent environmental pollution. The second records agreement with the statement that individual environmental action is pointless unless others act as well.

The script creates the cross-tabulation reported in the appendix and calculates a chi-squared test and Cramer's V. The chi-squared test asks whether the two responses are statistically independent. Cramer's V summarizes the strength of their association on a scale from 0 to 1.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "code" / "02_descriptives.py")], check=True)
pd.read_csv(TABLES / "table12_contingency_v199_v202.csv", index_col=0)

In [ ]:
print((TABLES / "chi2_cramersv.txt").read_text())

## 5. Estimate the ordered-response models

The paper uses ordered logistic regression because each outcome has three ranked categories: agree, neither agree nor disagree, and disagree. Both models include country, income-decile, and settlement-size indicators, together with the interpersonal-trust measure.

This step may take several minutes.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "code" / "03_ordered_logit.py")], check=True)

estimates = pd.read_csv(TABLES / "table5_ordered_logit_combined.csv", index_col=0)
estimates.tail(15)

## 6. Calculate average changes in predicted probabilities

Regression coefficients from an ordered model are difficult to interpret directly. The next step therefore reports how the predicted probability of each response category changes when income, settlement size, or trust changes, while the other observed characteristics are held fixed.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "code" / "04_marginal_effects.py")], check=True)
print((TABLES / "ame_trust.txt").read_text())

In [ ]:
pd.read_csv(TABLES / "table6_ame_population_uncertainty.csv").head(10)

## 7. Create the country-level figures

The final empirical step compares each country's share of respondents who reject the conditional-action statement with its interpersonal-trust share and mean income decile. Bubble size reflects the EVS population weight, and color identifies the European region.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "code" / "06_figures_empirical.py")], check=True)

for filename in ["Fig5_trust_vs_certainty.png", "Fig6_income_vs_certainty.png"]:
    path = FIGURES / filename
    if path.exists():
        display(Image(filename=str(path), width=1000))

## Data and output notes

- The analysis is deterministic: it uses no random sampling or simulation.
- The survey's original response codes and the exact recoding rules are documented in `code/config.py`.
- The descriptive analysis uses the 48,123-observation working sample. Responses outside the five substantive outcome categories are excluded from model estimation, leaving 43,001 observations.
- Generated tables are saved in `output/tables/`; generated figures are saved in `output/figures/`.